In [ ]:
import colorcet as cc
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats

import brain_probability_flux.plotting as bpp
from brain_probability_flux.dina.dynamics import ising as idyna
from brain_probability_flux.dina.network import visualization as netvis

In [ ]:
# Results are read from data/. To plot a fresh run, set DATA_DIR to its run
# directory, for example Path("../data/2026-10-20").
DATA_DIR = bpp.DATA_DIR

In [ ]:
max_fig_width = bpp.FIG_WIDTH
max_fig_height = bpp.FIG_HEIGHT

In [ ]:
pfa, cluster_inds = bpp.aggregated_pfa(data_dir=DATA_DIR)
num_clusters = len(np.unique(cluster_inds))
color = bpp.spin_colors(num_clusters)
pfa.perform_WPCA(spin_colors=color)

In [ ]:
flux_matrices = bpp.flux_matrices_by_task(pfa)
transition_indices = bpp.get_one_bit_difference_indices(num_clusters)

In [ ]:
# One Arrhenius network shared by all tasks, with a task-specific external
# input (scripts/infer_networks.py, group shared_network)
shared = bpp.load_results("inferred_networks", "shared_network", DATA_DIR)
beta_im, beta_ei = shared["beta_J"], shared["beta_h"]

# fig-s15

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(max_fig_width, max_fig_width / 3 - 0.5))
panels = {}
for name, ax, matrix, label in zip(
    "abc",
    axs,
    [beta_im, bpp.get_symmetric(beta_im), bpp.get_antisymmetric(beta_im)],
    ["Interaction", "Symmetric", "Antisymmetric"],
):
    fig, ax, image = netvis.plot_im(
        matrix,
        vmin=-1.002,
        vmax=1.002,
        plot_cbar=True,
        return_cbar=True,
        fig=fig,
        ax=ax,
        cmap=cc.cm.CET_D3_r,
    )
    bpp.color_cluster_ticks(ax, num_clusters)
    ax.set_xlabel(None)
    ax.set_ylabel(None)
    image.colorbar.set_label(label)
    panels[name] = [ax, image.colorbar.ax]
bpp.save_axes_panels(fig, panels, "../figures/fig-s15")

In [ ]:
fig, ax = plt.subplots(
    figsize=(max_fig_width / 3, max_fig_width / 3 + 0.1),
    sharex=True,
    sharey=True,
)
netvis.plot_network(
    beta_im,
    ax=ax,
    node_color=np.arange(7),
    cmap=plt.cm.tab10,
    vmin=0,
    vmax=9,
    node_size=60,
    plot_label=True,
    font_size=5,
    edge_width_scale=1,
)
ax.set_xlabel(None)
ax.set_ylabel(None)
ax.axis("off")
bpp.save_panel(fig, "../figures/fig-s15", "d", bbox_inches="tight")

In [ ]:
fig, axs = plt.subplots(
    3,
    3,
    figsize=(max_fig_width / 3, max_fig_width / 3 + 0.1),
    sharex=True,
    sharey=True,
)
for k, ax in enumerate(bpp.task_axes(axs)):
    bars = ax.bar(np.arange(num_clusters) + 1, beta_ei[k])
    for j, bar in enumerate(bars):
        bar.set_facecolor(f"C{j}")
    ax.axhline(0.0, c="k", ls="--")
    ax.set_title(pfa.task_name_list[k])
    ax.set_xticks(np.arange(num_clusters) + 1)
    ax.set_xticklabels(np.arange(num_clusters) + 1, fontweight="light")
    ax.set_ylim(-0.15, 0.15)
axs[0, 2].axis("off")
fig.supxlabel("Cluster ID")
fig.supylabel("External input")
bpp.save_panel(fig, "../figures/fig-s15", "e", bbox_inches="tight")

In [ ]:
fig, axs = plt.subplots(
    3, 3, figsize=(max_fig_width / 3, max_fig_width / 3 + 0.1)
)
bpp.network_grid(
    axs,
    [beta_im] * bpp.NUM_TASKS,
    beta_ei,
    pfa.task_name_list,
    cmap=cc.cm.CET_D3_r,
    vmin=-0.15,
    vmax=0.15,
    node_size=40,
    plot_label=True,
    font_size=5,
    font_color="k",
    font_weight="light",
    edge_width_scale=0.5,
)
bpp.save_panel(fig, "../figures/fig-s15", "f", bbox_inches="tight")

# fig-s16

In [ ]:
model_flux = [
    idyna.im_to_flux(beta_im, 1.0, magnetic_field=beta_ei[k])
    for k in range(bpp.NUM_TASKS)
]
r_values = [
    stats.pearsonr(
        flux_matrices[k][transition_indices], model_flux[k][transition_indices]
    ).statistic
    for k in range(bpp.NUM_TASKS)
]
fig, axs = plt.subplots(
    3, 3, figsize=(max_fig_width, max_fig_height), sharex=True, sharey=True
)
bpp.plot_pca_by_task(
    pfa,
    axs,
    scale=4,
    color=color,
    arrows=[idyna.flux_to_flux_list(flux) for flux in model_flux],
    titles=[
        f"{name}, $r={r:.2f}$" for name, r in zip(pfa.task_name_list, r_values)
    ],
)
bpp.save_panel(fig, "../figures/fig-s16", "fig-s16", bbox_inches=None)